In [ ]:
!wget http://cluster.ischool.drexel.edu/~jz85/SDSSLogViewer/sampledata/median.csv

--2026-03-18 12:40:38--  http://cluster.ischool.drexel.edu/~jz85/SDSSLogViewer/sampledata/median.csv
Resolving cluster.ischool.drexel.edu (cluster.ischool.drexel.edu)... 129.25.203.170
Connecting to cluster.ischool.drexel.edu (cluster.ischool.drexel.edu)|129.25.203.170|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 302235399 (288M) [text/plain]
Saving to: ‘median.csv.1’

median.csv.1          2%[                    ]   5.85M  2.17MB/s               ^C


--2026-03-26 08:33:34--  http://cluster.ischool.drexel.edu/~jz85/SDSSLogViewer/sampledata/median.csv
Resolving cluster.ischool.drexel.edu (cluster.ischool.drexel.edu)... ^C


In [1]:
!head -5 median.csv

yy,mm,dd,hh,mi,ss,seq,theTime,logID,clientIP,requestor,server,dbname,access,elapsed,busy,rows,statement,error,errorMessage,isvisible
2008,11,30,23,59,56,1767613780,11/30/2008 11:59:56 PM,8002,66.249.71.243,cas.sdss.org,SDSSSQL005,BestDR4,public,0.016,0,1,select * from Field where fieldId=0x08290cfd60860000  ,0,,1
2008,11,30,23,59,56,1767613779,11/30/2008 11:59:56 PM,8002,129.125.6.203,cas.sdss.org,SDSSSQL007,BestDR5,public,0.063,2E-3,431,"Select P.ra, P.dec, P.htmID, Q.objID,Q.photoz,Q.photozErr,Q.flag from PhotoObjAll P, Photoz2 Q where ((P.htmID >= 14504465268736 and P.htmID <= 14504469463039)) and P.objID = Q.objID  ",0,,1
2008,11,30,23,59,47,1767613781,11/30/2008 11:59:48 PM,8002,129.125.6.203,cas.sdss.org,SDSSSQL007,BestDR5,public,0.063,2E-3,338,"Select P.ra, P.dec, P.htmID, Q.objID,Q.photoz,Q.photozErr,Q.flag from PhotoObjAll P, Photoz2 Q where ((P.htmID >= 14504461074432 and P.htmID <= 14504465268735)) and P.objID = Q.objID  ",0,,1
2008,11,30,23,59,42,1767613786,11/30/2008 11:59

In [ ]:
import pandas as pd
df = pd.read_csv("median.csv", on_bad_lines='warn',encoding='latin1')

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
df.isna().sum(axis=1).value_counts().sort_index()

0      48983
1     777625
2          4
3       2319
4        138
12         1
13         1
14        15
15         5
16        15
17      1324
18      1235
19       116
20      6415
Name: count, dtype: int64

We are removing those which we have more than 2 null values because those are the querries where parsing have been failed we can reconstruct them by more careful inspection which we have tried but the amount of data we are getting back doesnt feel worth the work

In [4]:
df_cleaned=df[df.isna().sum(axis=1)<2]

In [5]:
colns=list(df_cleaned['error'].unique())

In [6]:
colns

[0.0,
 -1.0,
 '0',
 '-1',
 ' dec',
 '1',
 'dec FROM Star"',
 1.0,
 " s2.ew/(1+s2.z) as 'ew/(1+z)'",
 ' s2.lineID']

In [7]:
colns.remove(' dec')
colns.remove('dec FROM Star"')
colns.remove(" s2.ew/(1+s2.z) as 'ew/(1+z)'")
colns.remove(' s2.lineID')

In [8]:
df_cleaned=df_cleaned[df_cleaned['error'].isin(colns)]

In [9]:
# df_cleaned.errorMessage.unique()

We got the cleaned data 

In [10]:
import numpy as np
df_cleaned['errorMessage'].value_counts()

errorMessage
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [11]:
df_cleaned['error'] = pd.to_numeric(df_cleaned['error'], errors='coerce')

In [12]:
df_correct = df_cleaned[df_cleaned['error'] == 0]
print(df_correct.shape)

(817412, 21)


In [13]:
df_cleaned[df.error==1].sample(10)

/tmp/ipykernel_55/3752742740.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_cleaned[df.error==1].sample(10)


,yy,mm,dd,hh,mi,ss,seq,theTime,logID,clientIP,requestor,server,dbname,access,elapsed,busy,rows,statement,error,errorMessage,isvisible
220087,2008,11,21,3,49,25.0,1767114063.0,11/21/2008 3:49:26 AM,8002.0,202.141.155.253,,DR7Best long,BestDR7,casjobs,13447.600000,0.0,99999999.0,SELECT count(g.objID) into mydb.CutI from Phot...,1.0,,1.0
588406,2008,11.0,11,8.0,5.0,0.0,1767451361.0,11/11/2008 8:05:00 AM,8002.0,192.84.134.230,,DR5 long,BestDR5,casjobs,465.662994,0.0,99999999.0,"CREATE TABLE #UPLOAD(\n up_ra FLOAT,\t\t\t\t\t...",1.0,,1.0
717284,2008,11,5,13,51,48,1764216418,11/5/2008 1:51:49 PM,8002,200.14.68.146,,DR7Best long,BestDR7,casjobs,0.033000,0.0,99999999.0,"select s.plate, s.fiberid, s.mjd,t.run,t.rerun...",1.0,,1.0
585320,2008,11.0,11,11.0,41.0,30.0,1767448382.0,11/11/2008 11:41:31 AM,8002.0,131.225.7.9,,DR7Best,BestDR7,casjobs,0.063000,0.0,99999999.0,"select top 10 p.objid,p.run,p.rerun,p.camcol,p...",1.0,,1.0
612701,2008,11,10,9,43,27,1764119746.0,11/10/2008 9:43:27 AM,8002.0,192.167.203.2,,DR5 long,BestDR5,casjobs,0.140000,0.0,99999999.0,CREATE TABLE #UPLOAD ( \t\t\t\t\t\t\t\t\t\t\t\...,1.0,,1.0
402248,2008,11,16,16,58,32.0,1767279639.0,11/16/2008 4:58:32 PM,8002.0,132.229.224.39,,SQL010DBHost,mydb,casjobs,0.030000,0.0,99999999.0,drop table NGC362,1.0,,1.0
717863,2008,11,5,13,17,55,1764216938,11/5/2008 1:17:55 PM,8002,129.25.6.212,,DR7Collab long,BestDR7Collab,casjobs,0.016000,0.0,99999999.0,SELECT * into mydb.dr7qsoconcordanceall from q...,1.0,,1.0
716195,2008,11,5,14,30,24,1764215770,11/5/2008 2:30:24 PM,8002,128.95.99.173,,DR7Best long,BestDR7,casjobs,72.502998,0.0,99999999.0,"select into MyDB.spp_dr7 specobjid, plate, mjd...",1.0,,1.0
333204,2008,11.0,18,10.0,23.0,31.0,1767217676.0,11/18/2008 10:23:31 AM,8002.0,137.205.124.72,,DR6 long,BestDR6,casjobs,0.403000,0.0,99999999.0,"SELECT top 10000\npsfmag_u,psfmag_g,psfmag_r,p...",1.0,,1.0
404054,2008,11,16,15,43,42.0,1767281425.0,11/16/2008 3:43:42 PM,8002.0,132.229.224.39,,DR7Best,BestDR7,casjobs,0.110000,0.0,99999999.0,"SELECT plate,mjd,fiberid,specobjid, rva,rvaerr...",1.0,,1.0


In [14]:
statements=list(df_correct.statement)

In [15]:
import re
import sqlparse
from sqlparse.tokens import Number, String, Literal

def normalize_sql(sql):
    """Lowercase + collapse all whitespace into single spaces"""
    if not isinstance(sql, str):
        return ""
    sql = sql.lower().strip()
    sql = re.sub(r'\s+', ' ', sql)   
    return sql

def tokenize_sql(sql):
    """Robustly tokenize SQL and mask literals in one pass to prevent fragmentation"""
    sql = normalize_sql(sql)
    if not sql:
        return []
        
    parsed = sqlparse.parse(sql)
    if not parsed:
        return []
        
    tokens = []
    for token in parsed[0].flatten():
        if token.is_whitespace:
            continue
            
        if token.ttype in Number:
            tokens.append('NUM_LITERAL')
        elif token.ttype in String:
            tokens.append('STR_LITERAL')
        elif token.value.startswith('0x') and len(token.value) > 2:
            tokens.append('HEX_LITERAL')
        else:
            tokens.append(token.value)
            
    return tokens

In [16]:
def mask_literals(sql):
    return " ".join(tokenize_sql(sql))

In [17]:
sql = "select p.ra, p.dec from photoobjall p where p.htmid >= 14504465268736"
tokens = tokenize_sql(sql)
print(tokens)
print("scientfic literal check:", tokenize_sql("SELECT busy FROM logs WHERE busy > 2E-3"))

['select', 'p', '.', 'ra', ',', 'p', '.', 'dec', 'from', 'photoobjall', 'p', 'where', 'p', '.', 'htmid', '>=', 'NUM_LITERAL']
scientfic literal check: ['select', 'busy', 'from', 'logs', 'where', 'busy', '>', 'NUM_LITERAL']


In [18]:
!git clone https://github.com/SQL-Storm/SQLStorm.git

fatal: destination path 'SQLStorm' already exists and is not an empty directory.


In [19]:
import os 
os.chdir("SQLStorm")

In [20]:
ls v1.0/tpch

distinct_queries.csv  queries_generated/  queries.sql
queries/              queries_snowflake/


In [21]:
import pandas as pd
import os


def load_sql_dataframes(base_path):
    """
    Reads valid and invalid SQL query CSVs from the SQLStorm dataset,
    loads the corresponding .sql files, and returns two labeled DataFrames.

    Parameters
    ----------
    base_path : str
        Root path to the SQLStorm dataset (e.g., "/kaggle/working/SQLStorm/").

    Returns
    -------
    df_valid : pd.DataFrame
        DataFrame with columns ['file_name', 'sql_text', 'label'] where label=0.
    df_invalid : pd.DataFrame
        DataFrame with columns ['file_name', 'sql_text', 'label'] where label=1.
    """

    valid_csv_path = os.path.join(base_path, "v1.0", "stackoverflow", "valid_queries.csv")
    valid_queries_dir = os.path.join(base_path, "v1.0", "stackoverflow", "queries")

    valid_csv = pd.read_csv(valid_csv_path)
    valid_file_names = []
    valid_sql_texts = []

    for _, row in valid_csv.iterrows():
        file_name = str(row.iloc[0]) 
        sql_file_path = os.path.join(valid_queries_dir, file_name)

        try:
            with open(sql_file_path, "r", encoding="utf-8") as f:
                sql_text = f.read()
            valid_file_names.append(file_name)
            valid_sql_texts.append(sql_text)
        except FileNotFoundError:
            print(f"[SKIP] Valid query file not found: {sql_file_path}")
            continue

    invalid_csv_path = os.path.join(base_path, "v1.0", "stackoverflow", "invalid_queries.csv")
    invalid_queries_dir = os.path.join(base_path, "v1.0", "stackoverflow", "queries_generated")

    invalid_csv = pd.read_csv(invalid_csv_path)
    invalid_file_names = []
    invalid_sql_texts = []

    for _, row in invalid_csv.iterrows():
        file_name = str(row.iloc[0]) 
        sql_file_path = os.path.join(invalid_queries_dir, file_name)

        try:
            with open(sql_file_path, "r", encoding="utf-8") as f:
                sql_text = f.read()
            invalid_file_names.append(file_name)
            invalid_sql_texts.append(sql_text)
        except FileNotFoundError:
            print(f"[SKIP] Invalid query file not found: {sql_file_path}")
            continue

    df_valid = pd.DataFrame({
        "file_name": valid_file_names,
        "sql_text": valid_sql_texts,
        "label": 0
    })

    df_invalid = pd.DataFrame({
        "file_name": invalid_file_names,
        "sql_text": invalid_sql_texts,
        "label": 1
    })

    print(f"Loaded {len(df_valid)} valid queries   (label=0)")
    print(f"Loaded {len(df_invalid)} invalid queries (label=1)")

    return df_valid, df_invalid


In [22]:
df_valid_stack_overflow,df_invalid_stack_overflow=load_sql_dataframes("/kaggle/working/SQLStorm")

Loaded 12587 valid queries   (label=0)
Loaded 5664 invalid queries (label=1)


In [23]:
import pandas as pd
import os


def load_sql_dir(directory):
    """
    Load every .sql file in `directory` into a pandas DataFrame.

    Parameters
    ----------
    directory : str
        Path to the folder containing .sql files.

    Returns
    -------
    pd.DataFrame
        Columns: ['file_name', 'sql_text']
    """
    records = []

    for file_name in sorted(os.listdir(directory)):
        if file_name.lower().endswith(".sql"):
            file_path = os.path.join(directory, file_name)
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    sql_text = f.read()
                records.append({"file_name": file_name, "sql_text": sql_text})
            except (FileNotFoundError, PermissionError, UnicodeDecodeError) as e:
                print(f"[SKIP] {file_path} → {e}")

    df = pd.DataFrame(records, columns=["file_name", "sql_text"])
    print(f"Loaded {len(df)} .sql files from {directory}")
    return df
df_tpch=load_sql_dir("/kaggle/working/SQLStorm/v1.0/tpch/queries")

Loaded 17036 .sql files from /kaggle/working/SQLStorm/v1.0/tpch/queries


In [24]:
df_tpcds=load_sql_dir("/kaggle/working/SQLStorm/v1.0/tpcds/queries")

Loaded 15242 .sql files from /kaggle/working/SQLStorm/v1.0/tpcds/queries


In [25]:
df_job=load_sql_dir("/kaggle/working/SQLStorm/v1.0/job/queries")

Loaded 11714 .sql files from /kaggle/working/SQLStorm/v1.0/job/queries


In [26]:
df_sqlstorm=pd.concat([df_valid_stack_overflow,df_tpcds,df_tpch,df_job],axis=0)

In [27]:
df_sqlstorm.drop('label',axis=1,inplace=True)

In [28]:
statements=statements+list(df_sqlstorm['sql_text'])

In [29]:
statements[60000]

'SELECT TOP 100 p.ra,p.dec,p.type,p.modelmag_g,p.modelmag_r,p.modelmag_i,p.run,p.rerun,p.camcol,p.field,p.obj,p.objid,dbo.fIAUFromEq(p.ra,p.dec) FROM PhotoPrimary as p WHERE p.ra<191.629762020236 and p.ra>191.522370082318 and p.dec>52.07391331199533 and p.dec<52.14380994490558 order by p.modelmag_r  '

In [30]:
tokenize_sql(statements[60000])

['select',
 'top',
 'NUM_LITERAL',
 'p',
 '.',
 'ra',
 ',',
 'p',
 '.',
 'dec',
 ',',
 'p',
 '.',
 'type',
 ',',
 'p',
 '.',
 'modelmag_g',
 ',',
 'p',
 '.',
 'modelmag_r',
 ',',
 'p',
 '.',
 'modelmag_i',
 ',',
 'p',
 '.',
 'run',
 ',',
 'p',
 '.',
 'rerun',
 ',',
 'p',
 '.',
 'camcol',
 ',',
 'p',
 '.',
 'field',
 ',',
 'p',
 '.',
 'obj',
 ',',
 'p',
 '.',
 'objid',
 ',',
 'dbo',
 '.',
 'fiaufromeq',
 '(',
 'p',
 '.',
 'ra',
 ',',
 'p',
 '.',
 'dec',
 ')',
 'from',
 'photoprimary',
 'as',
 'p',
 'where',
 'p',
 '.',
 'ra',
 '<',
 'NUM_LITERAL',
 'and',
 'p',
 '.',
 'ra',
 '>',
 'NUM_LITERAL',
 'and',
 'p',
 '.',
 'dec',
 '>',
 'NUM_LITERAL',
 'and',
 'p',
 '.',
 'dec',
 '<',
 'NUM_LITERAL',
 'order by',
 'p',
 '.',
 'modelmag_r']

In [31]:
from joblib import Parallel, delayed
import os
from tqdm import tqdm
import json

def process_sql_parallel(sql):
    """Helper function for parallel execution"""
    try:
        if isinstance(sql, str) and len(sql.strip()) > 0:
            # tokenize_sql is already defined in the previous cell
            return tokenize_sql(sql)
        return None
    except Exception:
        return None

n_cores = max(1, os.cpu_count() - 1)
print(f"Parallel tokenization started using {n_cores} cores...")

# Run parallel processing
raw_results = Parallel(n_jobs=n_cores)(
    delayed(process_sql_parallel)(sql) for sql in tqdm(statements)
)

# Reconstruct the lists
tokenized_queries = []
failed_indices = []

for i, res in enumerate(raw_results):
    if res is None:
        tokenized_queries.append([])
        failed_indices.append(i)
    else:
        tokenized_queries.append(res)

print(f"\nTotal queries:     {len(statements)}")
print(f"Successfully tokenized: {len(statements) - len(failed_indices)}")
print(f"Failed:            {len(failed_indices)}")

# Save results
with open('tokenized_queries.json', 'w') as f:
    json.dump(tokenized_queries, f)
print("Done! Saved to 'tokenized_queries.json'")

Parallel tokenization started using 3 cores...


  0%|          | 0/873991 [00:00<?, ?it/s]

100%|██████████| 873991/873991 [19:19<00:00, 753.58it/s]  



Total queries:     873991
Successfully tokenized: 873991
Failed:            0
Done! Saved to 'tokenized_queries.json'


In [32]:
import json

with open('/kaggle/working/SQLStorm/tokenized_queries.json', 'r') as f:
    tokenized_queries = json.load(f)

# Verify
print(f"Total queries loaded: {len(tokenized_queries)}")
print(f"First query tokens:   {tokenized_queries[0][:15]}")
print(f"Token count of first: {len(tokenized_queries[0])}")

Total queries loaded: 873991
First query tokens:   ['select', '*', 'from', 'field', 'where', 'fieldid', '=', 'NUM_LITERAL']
Token count of first: 8


In [37]:
import torch
torch.save(tokenized_queries,'data_tokenised_queries.pt')

In [38]:
from collections import Counter

token_counts = Counter(tok for q in tokenized_queries for tok in q)

vocab_size = len(token_counts)
rare1 = sum(1 for t,c in token_counts.items() if c == 1)
rare2 = sum(1 for t,c in token_counts.items() if c <= 2)
rare5 = sum(1 for t,c in token_counts.items() if c <= 5)

print("Vocab size:", vocab_size)
print("Tokens with freq==1:", rare1, f"({rare1/vocab_size:.2%})")
print("Tokens with freq<=2:", rare2, f"({rare2/vocab_size:.2%})")
print("Tokens with freq<=5:", rare5, f"({rare5/vocab_size:.2%})")

Vocab size: 35022
Tokens with freq==1: 7797 (22.26%)
Tokens with freq<=2: 16809 (48.00%)
Tokens with freq<=5: 24145 (68.94%)


In [39]:
(1 for t,c in token_counts.items() if c == 1)

<generator object <genexpr> at 0x7816a38e5490>

In [ ]:
import re
from tqdm import tqdm

sneaky_number_pattern = re.compile(r'^[-+]?(?:\d+\.\d*|\d*\.\d+|\d+)(?:[eE][-+]?\d+)?$')

print("Sweeping dataset for sneaky unmasked numbers...")
fixed_tokenized_queries = []

for query in tqdm(tokenized_queries):
    cleaned_query = []
    for token in query:
        # If the token exactly matches a weird number format, mask it!
        if sneaky_number_pattern.fullmatch(token):
            cleaned_query.append('NUM_LITERAL')
        # Also catch pure integers that might have been attached to a minus sign
        elif re.fullmatch(r'^[-+]?\d+$', token):
            cleaned_query.append('NUM_LITERAL')
        else:
            cleaned_query.append(token)
            
    fixed_tokenized_queries.append(cleaned_query)

print("Done!")

Sweeping dataset for sneaky unmasked numbers...


100%|██████████| 873991/873991 [01:00<00:00, 14361.53it/s]

Done! ✅


In [41]:
from collections import Counter

token_counts = Counter(tok for q in fixed_tokenized_queries for tok in q)

vocab_size = len(token_counts)
rare1 = sum(1 for t,c in token_counts.items() if c == 1)
rare2 = sum(1 for t,c in token_counts.items() if c <= 2)
rare5 = sum(1 for t,c in token_counts.items() if c <= 5)

print("Vocab size:", vocab_size)
print("Tokens with freq==1:", rare1, f"({rare1/vocab_size:.2%})")
print("Tokens with freq<=2:", rare2, f"({rare2/vocab_size:.2%})")
print("Tokens with freq<=5:", rare5, f"({rare5/vocab_size:.2%})")

Vocab size: 35022
Tokens with freq==1: 7797 (22.26%)
Tokens with freq<=2: 16809 (48.00%)
Tokens with freq<=5: 24145 (68.94%)


In [45]:
els=tuple(t for t,c in token_counts.items() if c == 1)
els[:10]

('specobjidfrom',
 'specobji',
 'targphotoobj',
 'speczstatus',
 'fpritargetn',
 'ra15',
 'sbo',
 'fgetnearbyobjecteq',
 'objidt',
 'fgetnearbyphotoobjeq')

In [ ]:
from collections import Counter

token_counts = Counter(tok for q in tokenized_queries for tok in q)


rare_tokens = {t for t, c in token_counts.items() if c <= 2}

queries_with_rare = sum(1 for q in tokenized_queries if any(t in rare_tokens for t in q))
total_queries = len(tokenized_queries)
queries_removed = queries_with_rare
queries_kept = total_queries - queries_removed

print("Total queries:", total_queries)
print("Queries that would be removed (contain token freq<=2):", queries_removed)
print("Queries that would be kept:", queries_kept)
print("Removal %:", queries_removed / total_queries)

Total queries: 873991
Queries that would be removed (contain token freq<=2): 13337
Queries that would be kept: 860654
Removal %: 0.015259882538836212


In [ ]:
from collections import Counter

token_counts = Counter(tok for q in tokenized_queries for tok in q)
rare_tokens = {t for t, c in token_counts.items() if c <= 2}

tokenized_kept = [q for q in tokenized_queries if not any(t in rare_tokens for t in q)]

new_counts = Counter(tok for q in tokenized_kept for tok in q)

print("Original queries:", len(tokenized_queries))
print("Kept queries:", len(tokenized_kept))
print("Removed queries:", len(tokenized_queries) - len(tokenized_kept))

print("\nOriginal vocab size:", len(token_counts))
print("New vocab size after dropping:", len(new_counts))
print("Vocab reduction:", (1 - len(new_counts)/len(token_counts)))

Original queries: 873991
Kept queries: 860654
Removed queries: 13337

Original vocab size: 35022
New vocab size after dropping: 15614
Vocab reduction: 0.5541659528296499


In [ ]:
from collections import Counter


token_counts = Counter(tok for q in tokenized_kept for tok in q)
total_occ = sum(token_counts.values())

def stats_for_min_freq(min_freq):
    kept_vocab = {t for t,c in token_counts.items() if c >= min_freq}
    kept_occ = sum(c for t,c in token_counts.items() if c >= min_freq)
    unk_occ = total_occ - kept_occ

    return {
        "min_freq": min_freq,
        "kept_vocab": len(kept_vocab),
        "coverage": kept_occ/total_occ,
        "unk_rate": unk_occ/total_occ
    }

for mf in [2, 3, 5, 10]:
    s = stats_for_min_freq(mf)
    print(s)

{'min_freq': 2, 'kept_vocab': 15000, 'coverage': 0.9999869485432714, 'unk_rate': 1.3051456728564795e-05}
{'min_freq': 3, 'kept_vocab': 13678, 'coverage': 0.9999307465048507, 'unk_rate': 6.925349514929007e-05}
{'min_freq': 5, 'kept_vocab': 9318, 'coverage': 0.9996137066398564, 'unk_rate': 0.00038629336014366127}
{'min_freq': 10, 'kept_vocab': 6039, 'coverage': 0.9991540785472802, 'unk_rate': 0.000845921452719812}


In [ ]:
from collections import Counter
from tqdm import tqdm
import torch

MIN_FREQ = 5

token_counts = Counter(tok for q in tokenized_kept for tok in q)
kept_vocab = [t for t, c in token_counts.items() if c >= MIN_FREQ]
token2id = {"<PAD>": 0, "<START>": 1, "<UNK>": 2}

for token in sorted(kept_vocab):
    if token not in token2id:
        token2id[token] = len(token2id)

id2token = {idx: token for token, idx in token2id.items()}

print(f"Final Vocabulary Size: {len(token2id)}")
encoded_queries = []
UNK_ID = token2id["<UNK>"]

print("Encoding queries to integers...")
for query in tqdm(tokenized_kept):
    encoded_seq = [token2id.get(token, UNK_ID) for token in query]
    encoded_queries.append(encoded_seq)
print("\n--- Sanity Check ---")
print("Original tokens: ", tokenized_kept[0][:10])
print("Encoded IDs:     ", encoded_queries[0][:10])

torch.save({
    'encoded_queries': encoded_queries,
    'token2id': token2id,
    'id2token': id2token
}, 'final_encoded_data.pt')

print("\nSaved as 'final_encoded_data.pt'")

Final Vocabulary Size: 9321 🎯
Encoding queries to integers...


100%|██████████| 860654/860654 [00:10<00:00, 85667.77it/s] 



--- Sanity Check ---
Original tokens:  ['select', '*', 'from', 'field', 'where', 'fieldid', '=', 'NUM_LITERAL']
Encoded IDs:      [7178, 12, 3419, 3223, 9148, 3225, 52, 65]

✅ Saved as 'final_encoded_data.pt'


In [49]:
import torch

print("Loading data into memory...")
loaded_data = torch.load('final_encoded_data.pt')

encoded_queries = loaded_data['encoded_queries']
token2id = loaded_data['token2id']
id2token = loaded_data['id2token']

print("\nData successfully loaded!")
print("Total queries ready for AI:", len(encoded_queries))
print("Vocabulary size:", len(token2id))

print("\nFirst query (Integer IDs):", encoded_queries[0][:10])

Loading data into memory...

Data successfully loaded!
Total queries ready for AI: 860654
Vocabulary size: 9321

First query (Integer IDs): [7178, 12, 3419, 3223, 9148, 3225, 52, 65]
